# UAPOML - Week 2 Assignment
## Section A — Portfolio Fundamentals & Risk Metrics


### Problem 1


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# a) Create DataFrame and compute returns
data = {
    'RELIANCE': [2800, 2850, 2830, 2900, 2880, 2950],
    'INFY': [1450, 1470, 1460, 1490, 1510, 1500],
    'HDFCBANK': [1600, 1580, 1610, 1625, 1615, 1640],
    'TATAMOTORS': [520, 535, 528, 540, 555, 548]
}
prices_df = pd.DataFrame(data, index=['Day 1', 'Day 2', 'Day 3', 'Day 4', 'Day 5', 'Day 6'])
returns_df = prices_df.pct_change().dropna()
print("Daily Percentage Returns:")
display(returns_df)

# b) Compute units and daily portfolio value
weights = np.array([0.25, 0.25, 0.25, 0.25])
initial_investment = 1000000

day1_prices = prices_df.iloc[0].values
units = (initial_investment * weights) / day1_prices
print("\nUnits purchased on Day 1:")
print(pd.Series(units, index=prices_df.columns))

daily_portfolio_values = prices_df.dot(units)

plt.figure(figsize=(10, 5))
daily_portfolio_values.plot(title='Portfolio Value Over Time', marker='o')
plt.xlabel('Days')
plt.ylabel('Portfolio Value (in ₹)')
plt.legend(['Portfolio Value'])
plt.grid()
plt.show()

# c) Compute annualised volatility
daily_returns = daily_portfolio_values.pct_change().dropna()
daily_volatility = daily_returns.std()
annualised_volatility = daily_volatility * np.sqrt(252)
print(f"Portfolio Annualised Volatility: {annualised_volatility:.4f} ({annualised_volatility*100:.2f}%)")


### Problem 2


In [ ]:
# Simulate 50 days of portfolio returns
np.random.seed(42)
mu = 0.001
sigma = 0.015
simulated_returns = np.random.normal(mu, sigma, 50)

# a) VaR
var_95 = -np.percentile(simulated_returns, 5)
var_99 = -np.percentile(simulated_returns, 1)

print(f"VaR 95%: {var_95:.4f}")
print(f"VaR 99%: {var_99:.4f}")
print("Interpretation: A 95% VaR means there is a 5% chance the portfolio will lose more than this percentage in a single day.")

# b) CVaR
cvar_95 = -simulated_returns[simulated_returns < -var_95].mean()
cvar_99 = -simulated_returns[simulated_returns < -var_99].mean()
print(f"\nCVaR 95%: {cvar_95:.4f}")
print(f"CVaR 99%: {cvar_99:.4f}")
print("Interpretation: CVaR is more informative because it measures the average loss in the worst-case scenarios beyond the VaR threshold, capturing the severity of tail risk.")

# c) Max Drawdown
portfolio_values = 1000000 * (1 + simulated_returns).cumprod()
peak = np.maximum.accumulate(portfolio_values)
drawdown = (peak - portfolio_values) / peak
max_dd = drawdown.max()
print(f"\nMaximum Drawdown: {max_dd:.4f}")

trough_idx = drawdown.argmax()
peak_idx = portfolio_values[:trough_idx].argmax() if trough_idx > 0 else 0

plt.figure(figsize=(10, 5))
plt.plot(drawdown, label='Drawdown', color='red')
plt.axvspan(peak_idx, trough_idx, color='red', alpha=0.3, label='Max Drawdown Region')
plt.title('Portfolio Drawdown Over 50 Simulated Days')
plt.xlabel('Days')
plt.ylabel('Drawdown')
plt.legend()
plt.grid()
plt.show()


### Problem 3


In [ ]:
# 252 days of synthetic returns
simulated_returns_252 = np.random.normal(mu, sigma, 252)
Rf_annual = 0.06

# a) Sharpe Ratio
Rp_annual = simulated_returns_252.mean() * 252
sigma_p_annual = simulated_returns_252.std() * np.sqrt(252)
sharpe_ratio = (Rp_annual - Rf_annual) / sigma_p_annual

# b) Sortino Ratio
negative_returns = simulated_returns_252[simulated_returns_252 < 0]
sigma_d_annual = negative_returns.std() * np.sqrt(252)
sortino_ratio = (Rp_annual - Rf_annual) / sigma_d_annual

# c) Compare
comparison_df = pd.DataFrame({
    'Metric': ['Sharpe Ratio', 'Sortino Ratio'],
    'Value': [sharpe_ratio, sortino_ratio]
})
display(comparison_df)
print("Why Sortino penalises less: Sortino ignores upside volatility. High upside volatility increases standard deviation (penalizing Sharpe), but doesn't affect downside deviation.")
print("Comparison: Sortino is more appropriate for asymmetrical returns because it focuses purely on downside risk, whereas Sharpe penalizes beneficial upside volatility.")


### Problem 4


In [ ]:
np.random.seed(42)
eps = np.random.normal(0.001, 0.018, 200)
prices_200 = np.zeros(200)
prices_200[0] = 1000 * (1 + eps[0])
for t in range(1, 200):
    prices_200[t] = prices_200[t-1] * (1 + eps[t])
    
df_p4 = pd.DataFrame({'Price': prices_200})

# a) SMA Crossover
df_p4['SMA_10'] = df_p4['Price'].rolling(10).mean()
df_p4['SMA_30'] = df_p4['Price'].rolling(30).mean()

df_p4['Position'] = np.where(df_p4['SMA_10'] > df_p4['SMA_30'], 1, -1)
# Generate a signal column (+1 for long, -1 for short) shifted to avoid look-ahead bias
df_p4['Signal'] = df_p4['Position'].shift(1).fillna(0)

# b) Simulate
df_p4['Return'] = df_p4['Price'].pct_change()
df_p4['Strategy_Return'] = df_p4['Signal'] * df_p4['Return']

df_p4['Cum_BnH'] = 100000 * (1 + df_p4['Return']).cumprod()
df_p4['Cum_Strategy'] = 100000 * (1 + df_p4['Strategy_Return']).cumprod()

plt.figure(figsize=(10, 5))
plt.plot(df_p4['Cum_BnH'], label='Buy and Hold')
plt.plot(df_p4['Cum_Strategy'], label='Strategy')
plt.title('SMA Crossover Strategy vs Buy and Hold')
plt.xlabel('Days')
plt.ylabel('Portfolio Value')
plt.legend()
plt.grid()
plt.show()

# c) Win Rate and Profit Factor
winning_trades = (df_p4['Strategy_Return'] > 0).sum()
total_trades = (df_p4['Strategy_Return'] != 0).sum()
win_rate = winning_trades / total_trades if total_trades > 0 else 0

total_profit = df_p4.loc[df_p4['Strategy_Return'] > 0, 'Strategy_Return'].sum()
total_loss = np.abs(df_p4.loc[df_p4['Strategy_Return'] < 0, 'Strategy_Return'].sum())
profit_factor = total_profit / total_loss if total_loss > 0 else np.nan

print(f"Win Rate: {win_rate:.4f}")
print(f"Profit Factor: {profit_factor:.4f}")
print("\nDoes a Profit Factor > 1 guarantee a good strategy?")
print("No, a Profit Factor > 1 simply means gross profits exceed gross losses. It does not account for the frequency of trades, risk/drawdowns, transaction costs, and it might be heavily skewed by one or two lucky trades despite a low win rate.")


## Section B — Machine Learning Applications
### Problem 5


In [ ]:
# 300 days simulated price
np.random.seed(42)
eps_300 = np.random.normal(0.001, 0.018, 300)
prices_300 = np.zeros(300)
prices_300[0] = 500 * (1 + eps_300[0])
for t in range(1, 300):
    prices_300[t] = prices_300[t-1] * (1 + eps_300[t])

df_p5 = pd.DataFrame({'Price': prices_300})

# a) Engineer features
df_p5['Return_1d'] = df_p5['Price'].pct_change()
df_p5['SMA_5'] = df_p5['Price'].rolling(5).mean()
df_p5['SMA_20'] = df_p5['Price'].rolling(20).mean()
df_p5['Volatility_10'] = df_p5['Return_1d'].rolling(10).std()
df_p5['Momentum_5'] = df_p5['Price'] - df_p5['Price'].shift(5)

df_p5.dropna(inplace=True)
df_p5.reset_index(drop=True, inplace=True)

# b) Binary target variable
df_p5['Target'] = (df_p5['Return_1d'].shift(-1) > 0).astype(int)
df_p5.drop(df_p5.tail(1).index, inplace=True) # drop the last row where target is NaN

class_counts = df_p5['Target'].value_counts()
class_counts.plot(kind='bar', title='Class Balance')
plt.xlabel('Class (1: Up, 0: Down/Flat)')
plt.ylabel('Count')
plt.show()
print("Class Balance:\n", class_counts)

# c) Normalise
features = ['Return_1d', 'SMA_5', 'SMA_20', 'Volatility_10', 'Momentum_5']
X = df_p5[features].values
X_min = X.min(axis=0)
X_max = X.max(axis=0)
X_scaled = (X - X_min) / (X_max - X_min)

df_p5_scaled = pd.DataFrame(X_scaled, columns=features)
df_p5_scaled['Target'] = df_p5['Target'].values

print("\nWhy is feature scaling critical before applying KNN?")
print("KNN relies on distance metrics (like Euclidean distance). If features are on completely different scales (e.g. Prices vs Returns), features with large numerical ranges will artificially dominate the distance calculation, ignoring the contributions of smaller-scale features.")


### Problem 6


In [ ]:
# a) KNN from scratch
def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2)**2))

def knn_predict(X_train, y_train, X_test, k):
    y_pred = []
    for test_point in X_test:
        distances = [euclidean_distance(test_point, train_point) for train_point in X_train]
        nearest_indices = np.argsort(distances)[:k]
        nearest_labels = y_train[nearest_indices]
        majority_vote = 1 if np.sum(nearest_labels) > k/2 else 0
        y_pred.append(majority_vote)
    return np.array(y_pred)

# b) Predict and report accuracy
split_idx = int(len(df_p5_scaled) * 0.8)
X_train = df_p5_scaled[features].values[:split_idx]
y_train = df_p5_scaled['Target'].values[:split_idx]
X_test = df_p5_scaled[features].values[split_idx:]
y_test = df_p5_scaled['Target'].values[split_idx:]

k_values = [3, 5, 7, 11, 15]
accuracies = []

for k in k_values:
    preds = knn_predict(X_train, y_train, X_test, k)
    acc = np.mean(preds == y_test)
    accuracies.append(acc)
    print(f"k={k}, Accuracy: {acc:.4f}")

plt.figure(figsize=(8,4))
plt.plot(k_values, accuracies, marker='o')
plt.title('Accuracy vs k')
plt.xlabel('k')
plt.ylabel('Accuracy')
plt.grid()
plt.show()

best_k = k_values[np.argmax(accuracies)]
print(f"Optimal k: {best_k}")

# c) Confusion Matrix for best k
best_preds = knn_predict(X_train, y_train, X_test, best_k)
TP = np.sum((best_preds == 1) & (y_test == 1))
FP = np.sum((best_preds == 1) & (y_test == 0))
FN = np.sum((best_preds == 0) & (y_test == 1))
TN = np.sum((best_preds == 0) & (y_test == 0))

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
print(f"\nPrecision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print("\nWhich metric is more important when false positives are costly?")
print("Precision is more important. A costly false positive means we executed a trade expecting a positive return, but it resulted in a loss. Precision specifically measures how many of our predicted 'buy' signals were actually correct.")


### Problem 7


In [ ]:
# Linear Regression Target is next day return (continuous)
df_p5['Next_Return'] = df_p5['Return_1d'].shift(-1)
df_p5_reg = df_p5.dropna().copy()

X_reg = df_p5_reg[features].values
# Prepend bias
X_reg_bias = np.c_[np.ones(X_reg.shape[0]), X_reg]
y_reg = df_p5_reg['Next_Return'].values

split_idx_reg = int(len(df_p5_reg) * 0.8)
X_train_reg = X_reg_bias[:split_idx_reg]
y_train_reg = y_reg[:split_idx_reg]
X_test_reg = X_reg_bias[split_idx_reg:]
y_test_reg = y_reg[split_idx_reg:]

# a) Normal Equation
beta = np.linalg.inv(X_train_reg.T.dot(X_train_reg)).dot(X_train_reg.T).dot(y_train_reg)
print("Learned Coefficients (Normal Equation):")
print(f"Bias: {beta[0]:.6f}")
for i, f in enumerate(features):
    print(f"{f}: {beta[i+1]:.6f}")

print("\nInterpretation: A positive coefficient implies that an increase in the feature is associated with an expected increase in the next-day return. A negative coefficient implies an inverse relationship.")

# b) Predict and Compute MSE/R2
y_pred_reg = X_test_reg.dot(beta)
mse = np.mean((y_test_reg - y_pred_reg)**2)
r2 = 1 - (np.sum((y_test_reg - y_pred_reg)**2) / np.sum((y_test_reg - np.mean(y_test_reg))**2))

print(f"\nMSE: {mse:.6f}")
print(f"R² Score: {r2:.6f}")

plt.figure(figsize=(8,4))
plt.scatter(y_test_reg, y_pred_reg, alpha=0.5)
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], color='red', label='y=x')
plt.title('Actual vs Predicted Returns')
plt.xlabel('Actual Returns')
plt.ylabel('Predicted Returns')
plt.legend()
plt.grid()
plt.show()

# c) Gradient Descent
eta = 0.01
n_iterations = 1000
n = len(X_train_reg)

# Scale features for GD stability
X_train_gd = X_train_reg.copy()
X_train_gd[:, 1:] = (X_train_gd[:, 1:] - X_train_gd[:, 1:].mean(axis=0)) / X_train_gd[:, 1:].std(axis=0)

beta_gd = np.zeros(X_train_gd.shape[1])
losses = []

for _ in range(n_iterations):
    y_pred_gd = X_train_gd.dot(beta_gd)
    error = y_pred_gd - y_train_reg
    loss = np.mean(error**2)
    losses.append(loss)
    
    gradient = (1/n) * X_train_gd.T.dot(error) 
    beta_gd = beta_gd - eta * gradient

plt.figure(figsize=(8,4))
plt.plot(losses)
plt.title('Gradient Descent Loss Curve')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.grid()
plt.show()

print("Do the coefficients converge to the Normal Equation solution?")
print("Yes, because Mean Squared Error is a convex function, Batch Gradient Descent will naturally converge to the same global minimum as the Normal Equation (provided the learning rate is appropriate and features are scaled).")


### Problem 8


In [ ]:
# Problem 8 logic: Bridge ML predictions with portfolio construction.
# Simulate 300 days for the 4 assets based on the same random walk as P4/P5, using their Day 1 prices as P0.
np.random.seed(42)
simulated_assets = pd.DataFrame()
for col in prices_df.columns:
    P0 = prices_df[col].iloc[0]
    eps_asset = np.random.normal(0.001, 0.018, 300)
    prices = np.zeros(300)
    prices[0] = P0 * (1 + eps_asset[0])
    for t in range(1, 300):
        prices[t] = prices[t-1] * (1 + eps_asset[t])
    simulated_assets[col] = prices

# Engineer features for each asset and predict
predictions_list = []
test_actual_returns = []

for col in simulated_assets.columns:
    df_temp = pd.DataFrame({'Price': simulated_assets[col]})
    df_temp['Return_1d'] = df_temp['Price'].pct_change()
    df_temp['SMA_5'] = df_temp['Price'].rolling(5).mean()
    df_temp['SMA_20'] = df_temp['Price'].rolling(20).mean()
    df_temp['Volatility_10'] = df_temp['Return_1d'].rolling(10).std()
    df_temp['Momentum_5'] = df_temp['Price'] - df_temp['Price'].shift(5)
    
    df_temp.dropna(inplace=True)
    df_temp['Next_Return'] = df_temp['Return_1d'].shift(-1)
    df_temp.dropna(inplace=True)
    
    X_asset = df_temp[features].values
    X_asset_bias = np.c_[np.ones(X_asset.shape[0]), X_asset]
    
    split_idx_asset = int(len(df_temp) * 0.8)
    X_test_asset = X_asset_bias[split_idx_asset:]
    y_test_asset = df_temp['Next_Return'].values[split_idx_asset:]
    
    # Predict using beta from P7 Normal Equation
    y_pred_asset = X_test_asset.dot(beta)
    
    predictions_list.append(y_pred_asset)
    test_actual_returns.append(y_test_asset)

# a) Arrange predictions in a NumPy array
predictions_array = np.array(predictions_list).T # shape: (test_days, 4)
actual_returns_array = np.array(test_actual_returns).T

print("Shape of predictions array (test_days x 4_assets):", predictions_array.shape)

# b) Assign portfolio weights proportional to positive predicted returns
weights_list = []
for i in range(predictions_array.shape[0]):
    preds = predictions_array[i]
    positive_preds = np.maximum(preds, 0)
    total_positive = np.sum(positive_preds)
    
    if total_positive > 0:
        w = positive_preds / total_positive
    else:
        # Fallback if all predictions are negative - equal weight or 100% cash
        w = np.array([0.25, 0.25, 0.25, 0.25]) 
    weights_list.append(w)

weights_array = np.array(weights_list)
print("\nSum of weights for the first 5 days:", np.sum(weights_array[:5], axis=1))

# c) Backtest ML-driven vs Equal-weight
ml_portfolio_returns = np.sum(weights_array * actual_returns_array, axis=1)
eq_weights = np.array([0.25, 0.25, 0.25, 0.25])
eq_portfolio_returns = np.sum(eq_weights * actual_returns_array, axis=1)

ml_cum_returns = np.cumprod(1 + ml_portfolio_returns)
eq_cum_returns = np.cumprod(1 + eq_portfolio_returns)

plt.figure(figsize=(10, 5))
plt.plot(ml_cum_returns, label='ML-Driven Portfolio')
plt.plot(eq_cum_returns, label='Equal-Weight Portfolio')
plt.title('ML-Driven vs Equal-Weight Portfolio on Test Period')
plt.xlabel('Test Days')
plt.ylabel('Cumulative Return')
plt.legend()
plt.grid()
plt.show()

print("\nDoes the ML-driven portfolio outperform?")
print("This depends on the random seed and the learned coefficients. In pure random walks, it often underperforms or equals the baseline because future returns are essentially unpredictable noise.")
print("\nLimitation of this approach:")
print("It completely ignores risk (covariance) between the assets. A portfolio heavily weighted towards an asset with the highest predicted return might expose the investor to massive variance and drawdowns. Proper portfolio optimization (like mean-variance) should account for the covariance matrix.")


### Problem 9


In [ ]:
# a) k-Fold Cross Validation
def k_fold_cv(X, y, k_folds, k_neighbors):
    fold_size = len(X) // k_folds
    accuracies = []
    
    for i in range(k_folds):
        start = i * fold_size
        end = (i + 1) * fold_size if i != k_folds - 1 else len(X)
        
        X_test_fold = X[start:end]
        y_test_fold = y[start:end]
        
        X_train_fold = np.concatenate([X[:start], X[end:]])
        y_train_fold = np.concatenate([y[:start], y[end:]])
        
        preds = knn_predict(X_train_fold, y_train_fold, X_test_fold, k_neighbors)
        acc = np.mean(preds == y_test_fold)
        accuracies.append(acc)
        
    return np.mean(accuracies), np.std(accuracies)

k_knn_values = [3, 7, 11]
for k_val in k_knn_values:
    mean_acc, std_acc = k_fold_cv(X_scaled, df_p5['Target'].values, 5, k_val)
    print(f"k={k_val} | Mean Acc: {mean_acc:.4f} | Std Acc: {std_acc:.4f}")

# b) Summary DataFrame
summary_df = pd.DataFrame({
    'Model': ['KNN Classifier', 'Linear Regression'],
    'Accuracy/R²': [f"Acc: {np.max(accuracies):.4f}", f"R²: {r2:.4f}"],
    'MSE/N.A.': ['N.A.', f"MSE: {mse:.6f}"],
    'Best Param': [f"k={best_k}", "None"]
})
display(summary_df)

print("\nc) Which model would you deploy?")
print("I would likely deploy Linear Regression (or neither, given poor R2/Acc in random walks). LR is computationally lighter during live trading and explicitly maps relationships instead of relying on a highly dimensional space where KNN struggles. However, if KNN's accuracy significantly outperforms, it could be used for simple directional betting.")
print("\nTwo risks of deploying an ML model in a live portfolio:")
print("1. Overfitting: The model perfectly captures noise in the historical data rather than the signal, leading to catastrophic failure in live markets.")
print("2. Look-ahead bias: Using features that accidentally leak future data (e.g. using today's closing price to trade today's open), which makes backtests look perfect but live execution impossible.")


## Section C — Conceptual & Critical Thinking
### Problem 10


**a) Diversification**  
Mathematically, if the correlation $\rho_{ij}$ is low or negative, the covariance term $\sigma_i \sigma_j \rho_{ij}$ in the portfolio variance formula becomes small or negative. This reduces the total sum of the variance. Consequently, the overall portfolio risk ($\sigma_p$) ends up being substantially lower than the weighted average of the individual asset risks, effectively providing a "free lunch" of risk reduction without necessarily sacrificing expected returns.

**b) Technical vs. Fundamental Analysis in ML**  
One feature derived from fundamental analysis could be the **Price-to-Earnings (P/E) ratio**. To obtain and integrate it, we would fetch the historical P/E data via a fundamental data API (such as AlphaVantage or Bloomberg), align it with our daily price data using the date index (forward-filling if earnings are reported quarterly), and add it as a new column to our feature matrix `X`. We would then normalize it using Min-Max scaling alongside the technical features before training the KNN.

**c) The Curse of Dimensionality**  
As you add more features to KNN, the volume of the feature space increases exponentially, making the data extremely sparse. In this high-dimensional space, the distance between any two points tends to become roughly equal, meaning the concept of "nearest neighbours" loses its discriminative power. This severely degrades the model's performance in financial data. One technique to mitigate this is **Dimensionality Reduction**, such as applying Principal Component Analysis (PCA) or performing rigorous feature selection before training.

**d) Overfitting in Backtesting**  
1. **Data Snooping (Over-optimization):** Running thousands of model variations and picking the one that accidentally fitted the noise. Detect this by using a strictly separated Out-of-Sample (OOS) holdout set.  
2. **Look-ahead Bias:** Using data that wouldn't have been available at the time of the trade (e.g., using a daily close to execute a trade earlier that same day). Detect this by rigorously auditing feature timestamps.  
3. **Survivorship Bias:** Testing only on assets that currently exist, ignoring companies that went bankrupt during the period. Detect this by ensuring the backtesting universe includes historical constituent lists, including delisted tickers.

**e) Linear Regression Assumptions**  
1. **Volatility Clustering (Heteroskedasticity):** In financial time series, variance often changes over time (periods of high volatility clump together). This violates the assumption of constant variance, which distorts standard errors and makes confidence intervals unreliable.  
2. **Autocorrelation (Serial Correlation):** Financial returns or their residuals often exhibit correlation with their own past values. This violates the assumption of independent residuals, leading the model to underestimate true variance and overstate its predictive confidence.
